In [1]:
import os
import sys
import copy
import torch
import argparse
from pprint import pprint


# Import ssu packages
sys.path.append('../src')
sys.path.append('../config')
# config packages
import read_config
# src packages
from eval import ABC_eval
from models import unet
from models import model as simpleModels
from models import unet as unetModels
from training import model_training_v2 as model_training
from logger import wandb_logging
from data_loader import ABC_dataset_loader
from utils import fvdb_utils as fu
from utils import ssu_tools as st 

import torch
import gc

def clear_gpu_memory():
    """Clear all GPU memory allocated by PyTorch"""
    
    if torch.cuda.is_available():
        # Clear PyTorch cache
        torch.cuda.empty_cache()
        
        # Force garbage collection
        gc.collect()
        
        # Clear all tensors from GPU
        torch.cuda.synchronize()
        
        # Get memory info
        allocated = torch.cuda.memory_allocated() / 1e9
        cached = torch.cuda.memory_reserved() / 1e9
        
        print(f"GPU Memory - Allocated: {allocated:.2f} GB, Cached: {cached:.2f} GB")
        
        # If memory is still allocated, try more aggressive clearing
        if allocated > 0:
            print("Attempting to clear more GPU memory...")
            print("Avoiding this step")
            # torch.cuda.empty_cache()
            # torch.cuda.synchronize()
            # gc.collect()
            
        print("✅ GPU memory cleared")
    else:
        print("❌ No CUDA GPU available")

# Call this function
clear_gpu_memory()

# def main(config_file):
config_file = 'config_55_10102025_1400.yaml'
# read config file
config = read_config.read_yaml_config(f'{config_file}')

print("Configuration loaded:")
for key, value in config.items():
    pprint(f"{key}: {value}")

# initialize logging
logger = wandb_logging.WandbLogger(
                    logging=config['logging'],
                    project_name=config['wandb']['project_name'],
                    entity=config['wandb']['entity'],
                    name=config['wandb']['name'],
                    group=config['wandb']['group'],
                    tags=config['wandb']['tags'],
                    notes=config['wandb']['notes'],
                    config=config['wandb']['config'],
                    # resume="allow",
                    # id = 'z3zwr3y6'
                )
logger.update_config('config_file_name', config_file)

# set reproducibility
st.set_reproducibility(is_reproducible=config['reproducibility']['is_reproducible'],
                        seed=config['reproducibility']['seed'])

# load data
input_dir = config['data']['input_dir']
names_set = os.listdir('/data/workspaces/spanwar/dataset/preprocessing_nmc_data/data_preprocessing/get_groundtruth_NMC/gt')


### if this work ###
# 1. Add gaussian noise to the input
# 2. change the normalization
# 3. Increae the steps 10 to 20
# 4. use dataprocessing
dataLoader = ABC_dataset_loader.ABCDataLoader(
                                    input_dir=input_dir,
                                    config=config,
                                    n_samples=10
                                )
(train_dataloader, 
val_dataloader, 
test_dataloader) = dataLoader.get(names_set=names_set)
logger.update_config('data_size', len(os.listdir(input_dir)))

GPU Memory - Allocated: 0.00 GB, Cached: 0.00 GB
✅ GPU memory cleared
Configuration loaded:
'logging: True'
"reproducibility: {'is_reproducible': True, 'seed': 42}"
("data: {'input_dir': "
 "'/data/workspaces/spanwar/dataset/preprocessing_nmc_data/data_preprocessing/get_groundtruth_NMC/gt_large', "
 "'data_description': 'Generated SDF of 32, 64, 128, 256 data using NMC "
 "logic', 'data_name': 'V1_Manual_Extracted_NMC_Data_large', 'data_split': "
 "{'train': 0.6, 'val': 0.2, 'test': 0.2}, 'dataset_grids': [32, 64, 128], "
 "'mask_threshold': {32: 65, 64: 129}, 'sdf_scaling_value': {32: 65, 64: 129}, "
 "'input_size': {32: 33, 64: 65}, 'unique_random_direction': True, 'is_crop': "
 "{'train': False, 'val': False, 'test': False}, 'crops_ratio': [1, 0.5, "
 "0.25], 'crops_threshold': {32: {1: 1600, 0.5: 800, 0.25: 400}, 64: {1: 5600, "
 "0.5: 2800, 0.25: 1400}}, 'n_jobs': -1, 'batch_size': 16, 'shuffle': "
 "{'train': True, 'val': False, 'test': False}, 'num_workers': 0}")
("training: {'u

100%|██████████| 2/2 [00:00<00:00, 1903.91it/s]


In [2]:
batch = next(iter(train_dataloader))
names, input_size, input_vdb, output_vdb = batch
# (obj_names, 
#                  vdb_input_32, 
#                  vdb_input_64, 
#                  new_ijkss_32, 
#                  new_ijkss_64, 
#                  new_featuress_32, 
#                  new_featuress_64, 
#                  actual_sdfs) = batch

In [3]:
input_size

[33, 33, 33, 33, 33, 33]

In [4]:
input_vdb[0].jdata

tensor([[ 2.7381, -1.0000,  0.5000,  0.5000],
        [ 2.7105, -0.5000, -0.5000,  1.0000],
        [ 2.7105, -0.5000,  1.0000,  0.5000],
        ...,
        [ 2.7223,  1.0000,  0.0000,  0.0000],
        [ 2.5176,  0.5000, -1.0000,  0.0000],
        [ 2.7381,  0.0000,  1.0000,  0.0000]], device='cuda:0')

In [5]:
from meshplot import plot
def plot_vdb(vdb):
    v, f, _ = vdb.grid.marching_cubes(vdb.data.jdata[:, 0])
    v = v.jdata.detach().cpu().numpy()
    f = f.jdata.detach().cpu().numpy()
    plot(v,f)
plot_vdb(input_vdb[-1])

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.5, 0.5,…

# check the prediction

In [1]:
import os
import fvdb
from meshplot import plot   

In [2]:
import torch
m = torch.nn.ReLU()
input = torch.tensor([0.1])
output = m(input)
output

tensor([0.1000])

In [3]:
import trimesh

def plot_vdb_with_grid_and_data(grid, data, is_save=False):
    v, f, _ = grid.marching_cubes(data)
    v = v.jdata.detach().cpu().numpy()
    f = f.jdata.detach().cpu().numpy()
    plot(v,f)
    if is_save:
        mesh = trimesh.Trimesh(vertices=v, faces=f)
        mesh.export('data/64_128_00000020.ply')
         

In [4]:
# path = '/data/workspaces/spanwar/results/ssu/test_predictions/55_improve_49_with_SDG_and_scheduler_and_64'
path = '/data/workspaces/spanwar/results/ssu/test_predictions/66_test'
path = '/data/workspaces/spanwar/results/ssu/test_predictions/66_with_custom_loss_per_change'

In [5]:
sorted(os.listdir(path))

['32_00000297.nvdb',
 '32_00000575.nvdb',
 '32_00000980.nvdb',
 '32_00001475.nvdb',
 '32_00001559.nvdb',
 '32_00001749.nvdb',
 '32_00001855.nvdb',
 '32_00002426.nvdb',
 '32_00002515.nvdb',
 '32_00002580.nvdb',
 '32_00002965.nvdb',
 '32_00003557.nvdb',
 '32_00003966.nvdb',
 '32_00004199.nvdb',
 '32_00004774.nvdb',
 '32_00004800.nvdb',
 '32_00004902.nvdb',
 '32_00004932.nvdb',
 '32_00005082.nvdb',
 '32_00005112.nvdb',
 '32_00005724.nvdb',
 '32_00005754.nvdb',
 '32_00005836.nvdb',
 '32_00006007.nvdb',
 '32_00006316.nvdb',
 '32_00006658.nvdb',
 '32_00006766.nvdb',
 '32_00007223.nvdb',
 '32_00007265.nvdb',
 '32_00007777.nvdb',
 '32_00007792.nvdb',
 '32_00007808.nvdb',
 '32_00007826.nvdb',
 '32_00007913.nvdb',
 '32_00008322.nvdb',
 '32_00009041.nvdb',
 '32_00009354.nvdb',
 '32_00009423.nvdb',
 '32_00009515.nvdb',
 '32_00009846.nvdb',
 '64_00000297.nvdb',
 '64_00000575.nvdb',
 '64_00000980.nvdb',
 '64_00001475.nvdb',
 '64_00001559.nvdb',
 '64_00001749.nvdb',
 '64_00001855.nvdb',
 '64_00002426

In [6]:
# 32_00000544
# sorted(os.listdir(path))

In [7]:
path = '/data/workspaces/spanwar/results/ssu/test_predictions/66_with_custom_loss_per_change'
grid, data, _ = fvdb.load(os.path.join(path, '32_00000575.nvdb'))
plot_vdb_with_grid_and_data(grid, data)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.5000909…

In [8]:
import h5py
import trimesh
gt_large_path = '/data/workspaces/spanwar/dataset/preprocessing_nmc_data/data_preprocessing/get_groundtruth_NMC/gt_large'
file_name = '00000020'
file_path = os.path.join(gt_large_path, f'{file_name}.hdf5')
gt_obj_path = f'/data/workspaces/spanwar/dataset/preprocessing_nmc_data/abc_dataset_objs/{file_name}/model.obj'
with h5py.File(file_path, 'r') as f:
    # Read specific resolutions
    sdf_128 = f['128_sdf'][:] # 128³ resolution
    sdf_32 = f['32_sdf'][:]

import numpy as np
from skimage import measure
import meshplot as mp

# Get vertices and faces using marching cubes
vertices, faces, normals, values = measure.marching_cubes(sdf_32, level=0)
# Plot the mesh
mp.plot(vertices, faces)

# Get vertices and faces using marching cubes
vertices, faces, normals, values = measure.marching_cubes(sdf_128, level=0)
# Plot the mesh
mp.plot(vertices, faces)

# display fvdb marching cube
import sys
sys.path.append('../src')
from utils import mesh_tools  as mt 
import fvdb.nn as fvnn
grid_33 = mt.mesh_grid(33)
grid_33 = torch.tensor(grid_33)
# Create grid from coordinates
grid = fvdb.gridbatch_from_ijk(
    fvdb.JaggedTensor(grid_33),  # grid coordinates
    voxel_sizes=1/32,  # size of each voxel (33-1 = 32 divisions)
    origins=torch.tensor([0, 0, 0])  # origin of the grid
)
ijk = grid.ijk.jdata
# Create FVDB tensor from grid and SDF data
fvdb_tensor = fvnn.VDBTensor(
    grid,
    grid.jagged_like(torch.tensor(sdf_32[ijk[:, 0], ijk[:, 1], ijk[:, 2]][:, None]))  # Add channel dimension
)

v, f, _ = fvdb_tensor.grid.marching_cubes(fvdb_tensor.data)
v = v.jdata.detach().cpu().numpy()
f = f.jdata.detach().cpu().numpy()
plot(v, f)

mesh = trimesh.load(gt_obj_path)
plot(mesh.vertices, mesh.faces)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(16.0, 16.…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(64.0, 64.…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.5, 0.50…

/user/spanwar/home/.conda/envs/fvdb_ponq/lib/python3.11/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "float32" does not match required type "float64". A coerced copy has been created.
  warnings.warn(
/user/spanwar/home/.conda/envs/fvdb_ponq/lib/python3.11/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "uint32" does not match required type "float64". A coerced copy has been created.
  warnings.warn(


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.0, 0.0,…

In [9]:
# 32_00004902
# 32_00000003
path = '/data/workspaces/spanwar/results/ssu/test_predictions/66_test'
grid, data, _ = fvdb.load(os.path.join(path, '64_00000020.nvdb'))
plot_vdb_with_grid_and_data(grid, data)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.5004497…

In [16]:
# 32_00004902
path = '/data/workspaces/spanwar/results/ssu/test_predictions/66_original'
grid, data, _ = fvdb.load(os.path.join(path, '32_00000003.nvdb'))
plot_vdb_with_grid_and_data(grid, data)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.5007826…

In [17]:
path = '/data/workspaces/spanwar/results/ssu/test_predictions/66_original_improve_inference'
grid, data, _ = fvdb.load(os.path.join(path, '32_00000003.nvdb'))
plot_vdb_with_grid_and_data(grid, data)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.5001029…

In [56]:
path = '/data/workspaces/spanwar/results/ssu/test_predictions/59_rerun_55_with_mse_sign_loss'
grid, data, _ = fvdb.load(os.path.join(path, '32_00000297.nvdb'))
plot_vdb_with_grid_and_data(grid, data)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.4992069…

In [11]:
path = '/data/workspaces/spanwar/results/ssu/test_predictions/66_test_weighted_loss_e_1'
grid, data, _ = fvdb.load(os.path.join(path, '32_00000020.nvdb'))
plot_vdb_with_grid_and_data(grid, data)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.5003893…

In [11]:
grid, data, _ = fvdb.load(os.path.join(path, '32_00000020.nvdb'))
plot_vdb_with_grid_and_data(grid, data)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.4996708…

In [12]:
grid, data, _ = fvdb.load(os.path.join(path, '64_00000020.nvdb'))
plot_vdb_with_grid_and_data(grid, data, is_save=True)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.4998073…

In [33]:
# display the ply file  
import trimesh
from meshplot import plot
mesh = trimesh.load('/user/spanwar/home/Documents/learn-fvdb/ssu/SSU/comparision/reach_for_the_spheres.ply')
plot(mesh.vertices, mesh.faces)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-0.000373…

In [18]:
mesh = trimesh.load('/user/spanwar/home/Documents/learn-fvdb/ssu/SSU/comparision/reach_for_the_arcs.ply')
plot(mesh.vertices, mesh.faces)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-0.000292…

In [19]:
719/60

11.983333333333333

# Test with PoNQ

In [39]:
obj_names = ['00009954', '00009941', '00009956', '00009913', '00009982', '00009904']
path = '/data/workspaces/spanwar/results/ssu/test_predictions/58_just_rerun_55'
for obj_name in obj_names:
    org_obj_path = f"/data/workspaces/spanwar/dataset/preprocessing_nmc_data/abc_dataset_objs/{obj_name}/model.obj"
    org_mesh = trimesh.load(org_obj_path)
    plot(org_mesh.vertices, org_mesh.faces)
    grid, data, _ = fvdb.load(os.path.join(path, f'64_{obj_name}.nvdb'))
    plot_vdb_with_grid_and_data(grid, data)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.0, 0.0,…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.5000640…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.0, 0.0,…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.4997190…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.0, 0.0,…

...

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.5004816…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.0, 0.0,…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.5004043…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.0, 1.49…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.5003022…